# OFI cancel — net PnL per fill across spread regimes

Visualisation of the finding from `bpt-research/findings/2026-05-24_ofi_cancel_spread_aware.md`.

**Headline:** the OFI-cancel rule lifts mean net PnL per fill by a constant **+0.915 bps**, *independent of spread regime*. Cancellation reduces toxic-flow cost by a fixed amount regardless of how much spread AS earns.

**Setup:** the sweep data is inlined below — no canon files, no CSV, no LD_PRELOAD required. Open → Run All → figures render.

To regenerate the underlying sweep from new canon days, see `bpt-research/experiments/ofi_cancel_spread_aware.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sweep result from ofi_cancel_spread_aware.py run on 10 HL hour-00 canon
# days, 122 instruments, 732,395 trades. Markout horizon = 1s. Values
# transcribed from findings/2026-05-24_ofi_cancel_spread_aware.md.
THETA = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, float('inf')]
SPREAD = [0.0, 2.0, 4.0, 6.0, 10.0]
N_INSTRUMENTS = 122

# Mean net PnL per fill (bps), rows=spread, cols=theta
mean_pnl = np.array([
    [-2.060, -2.420, -2.640, -2.850, -2.950, -2.970, -2.975],  # spread 0
    [-1.060, -1.420, -1.640, -1.850, -1.950, -1.970, -1.975],  # spread 2 ← HL maker
    [-0.060, -0.420, -0.640, -0.850, -0.950, -0.970, -0.975],  # spread 4
    [+0.940, +0.580, +0.360, +0.150, +0.050, +0.030, +0.025],  # spread 6
    [+2.940, +2.580, +2.360, +2.150, +2.050, +2.030, +2.025],  # spread 10
])

# Pos-EV instrument count out of 122, rows=spread, cols=theta
pos_count = np.array([
    [  1,   0,   0,   0,   0,   0,   0],  # spread 0
    [ 46,  27,  20,   9,   7,   7,   7],  # spread 2
    [ 82,  72,  65,  55,  54,  52,  52],  # spread 4
    [102,  95,  92,  88,  85,  85,  85],  # spread 6
    [112, 109, 108, 106, 106, 105, 105],  # spread 10
])

pivot_mean = pd.DataFrame(mean_pnl, index=SPREAD, columns=THETA)
pivot_mean.index.name = 'spread_bps'
pivot_mean.columns.name = 'theta_sigma'

pivot_pos_frac = pd.DataFrame(pos_count, index=SPREAD, columns=THETA) / N_INSTRUMENTS
pivot_pos_frac.index.name = 'spread_bps'
pivot_pos_frac.columns.name = 'theta_sigma'

def col_label(c):
    return '∞' if np.isinf(c) else f'{c:.1f}σ'

## Table 1 — mean net PnL per fill (bps), spread × θ

The **2.0 bps row is HL's typical maker spread**; the **∞ column is the no-cancel baseline**.

In [ ]:
pivot_mean.round(3)

## Heatmap — mean net PnL per fill

Red = AS loses money per fill. Blue = AS earns money per fill. The vertical distance from any θ column to the ∞ column is the cancel-rule's uplift — visibly constant across rows.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
vmax = np.nanmax(np.abs(pivot_mean.values))
im = ax.imshow(pivot_mean.values, aspect='auto', cmap='RdBu',
                vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(pivot_mean.columns)))
ax.set_xticklabels([col_label(c) for c in pivot_mean.columns])
ax.set_yticks(range(len(pivot_mean.index)))
ax.set_yticklabels([f'{s:.1f}' for s in pivot_mean.index])
ax.set_xlabel('cancel threshold θ')
ax.set_ylabel('maker spread (bps)')
ax.set_title('Mean net PnL per fill (bps) — spread × θ')
for i in range(pivot_mean.shape[0]):
    for j in range(pivot_mean.shape[1]):
        v = pivot_mean.iat[i, j]
        ax.text(j, i, f'{v:+.2f}', ha='center', va='center',
                color='white' if abs(v) > vmax * 0.5 else 'black',
                fontsize=9)
fig.colorbar(im, ax=ax, label='bps/fill')
plt.tight_layout()
plt.show()

## The +0.915 bps uplift, visualised

For each spread regime, compute `(best θ PnL) − (no-cancel PnL)`. The signature claim is that this delta is **invariant in spread** — should be a flat horizontal line at +0.915.

In [ ]:
no_cancel = pivot_mean[float('inf')]
best_per_spread = pivot_mean.max(axis=1)
uplift = (best_per_spread - no_cancel).rename('uplift_bps')

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(len(uplift)), uplift.values, color='steelblue')
ax.set_xticks(range(len(uplift)))
ax.set_xticklabels([f'{s:.0f}' for s in uplift.index])
ax.set_xlabel('maker spread (bps)')
ax.set_ylabel('best-θ vs no-cancel uplift (bps/fill)')
ax.set_title('Cancel rule uplift is invariant across spread regimes')
for i, v in enumerate(uplift.values):
    ax.text(i, v + 0.02, f'+{v:.3f}', ha='center', fontsize=10)
ax.axhline(uplift.mean(), color='red', linestyle='--', linewidth=1,
            label=f'mean = +{uplift.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

print('Per-spread uplift table:')
print(pd.DataFrame({'no_cancel': no_cancel,
                    'best_theta': best_per_spread,
                    'uplift': uplift}).round(3))

## Net PnL vs θ, one line per spread

Reads left to right as: "as you tighten the cancel rule (lower θ ⇒ more aggressive cancelling), what happens to net PnL?" Each spread regime shifts the curve up by a fixed amount but otherwise has the *same* shape — confirming the additivity of spread revenue and toxic-flow saving.

In [ ]:
finite_theta = [t for t in THETA if not np.isinf(t)]

fig, ax = plt.subplots(figsize=(8, 4.5))
for sp in pivot_mean.index:
    y = pivot_mean.loc[sp, finite_theta].values
    label = f'{sp:.0f} bps'
    if sp == 2.0:
        label += '  ← HL maker'
    ax.plot(finite_theta, y, marker='o', label=label,
            linewidth=2 if sp == 2.0 else 1)

for sp in pivot_mean.index:
    ax.axhline(no_cancel[sp], linestyle=':', alpha=0.3, color='gray')

ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('cancel threshold θ (σ units)')
ax.set_ylabel('mean net PnL per fill (bps)')
ax.set_title('Net PnL per fill vs cancel aggressiveness, by spread regime')
ax.legend(title='maker spread', loc='center right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Fraction of profitable instruments, spread × θ

The **mean** PnL story understates the rule's value. The more interesting question for deployment is: how many instruments flip from negative-EV to positive-EV under cancellation? At HL's 2 bps spread, cancellation lifts the positive-EV universe from **6% (7/122) → 38% (46/122)** — a 6× expansion of deployable instruments. *That's* the load-bearing finding.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot_pos_frac.values, aspect='auto', cmap='YlGn',
                vmin=0, vmax=1)
ax.set_xticks(range(len(pivot_pos_frac.columns)))
ax.set_xticklabels([col_label(c) for c in pivot_pos_frac.columns])
ax.set_yticks(range(len(pivot_pos_frac.index)))
ax.set_yticklabels([f'{s:.1f}' for s in pivot_pos_frac.index])
ax.set_xlabel('cancel threshold θ')
ax.set_ylabel('maker spread (bps)')
ax.set_title(f'Fraction of profitable instruments (n={N_INSTRUMENTS} per cell)')
for i in range(pivot_pos_frac.shape[0]):
    for j in range(pivot_pos_frac.shape[1]):
        v = pivot_pos_frac.iat[i, j]
        ax.text(j, i, f'{v:.0%}', ha='center', va='center',
                color='white' if v > 0.5 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, label='fraction')
plt.tight_layout()
plt.show()

## Summary — what this notebook says

1. **Uplift is +0.915 bps/fill, invariant in spread regime.** The bar chart is the headline figure.
2. **At HL's 2 bps maker spread, mean PnL is still negative** even with the best θ (−1.06 bps/fill). The strategy needs *more* than this rule to break even on the average instrument.
3. **The positive-EV universe expands 6×** under cancellation at spread=2 bps (7 → 46 instruments). That's the practical lever — restrict the universe + apply the rule.
4. **Next:** port the rule to `bpt-strategy/src/strategy/avellaneda_stoikov_strategy.cpp` with `ofi_cancel_threshold_sigma` config, then run through `bpt-backtester` for the realistic queue + inventory interaction the per-fill sim can't model.